# 🧠⚖️ Magistrado Themis — Fine-Tuning Pipeline
## GATE 1: Dataset Forense + GATE 2: QLoRA sobre Qwen-2.5-3B

**Propriedade Intelectual:** Symbeon Labs  
**Classificação:** `CONFIDENCIAL — SEGREDO INDUSTRIAL`

---

### ⚙️ Configuração Recomendada
- **Runtime:** GPU → T4 (gratuito) ou A100 (Pro+)
- **RAM:** mínimo 12GB
- **Disco:** mínimo 15GB livres

### 📋 Pipeline
```
0. Verificar GPU
1. Instalar dependências
2. Clonar repositório (privado)
3. GATE 1 — Gerar dataset forense
4. GATE 2 — Fine-tuning QLoRA
5. Testar modelo treinado
6. Salvar e baixar modelo
```

> ⚠️ **Execute as células em ordem.** Cada etapa depende da anterior.

---
## 🔍 Etapa 0 — Verificar GPU e Ambiente

In [ ]:
import torch
import os

print('='*60)
print('  MAGISTRADO THEMIS — Environment Check')
print('='*60)

# GPU
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu}')
    print(f'✅ VRAM: {vram:.1f} GB')
    if vram < 8:
        print('⚠️  AVISO: VRAM < 8GB. Reduza batch_size para 1.')
else:
    print('❌ ERRO: GPU não detectada!')
    print('   Vá em: Runtime → Change Runtime Type → GPU')
    raise RuntimeError('GPU necessária para fine-tuning QLoRA')

# Disco
disk = os.popen('df -h /').read().split('\n')[1].split()
print(f'✅ Disco disponível: {disk[3]}')

# Python
import sys
print(f'✅ Python: {sys.version.split(" ")[0]}')
print('='*60)

---
## 📦 Etapa 1 — Instalar Dependências

In [ ]:
# ⏱️ ~3-5 minutos na primeira execução
print('📦 Instalando dependências do pipeline QLoRA...')

!pip install -q \
    transformers==4.44.2 \
    peft==0.12.0 \
    trl==0.11.4 \
    bitsandbytes==0.43.3 \
    datasets==3.0.1 \
    accelerate==0.34.2 \
    einops \
    scipy

print('✅ Dependências instaladas!')

In [ ]:
# Verifica versões instaladas
import transformers, peft, bitsandbytes, datasets, accelerate

print(f'✅ transformers: {transformers.__version__}')
print(f'✅ peft:         {peft.__version__}')
print(f'✅ bitsandbytes: {bitsandbytes.__version__}')
print(f'✅ datasets:     {datasets.__version__}')
print(f'✅ accelerate:   {accelerate.__version__}')

---
## 🔐 Etapa 2 — Autenticação e Clonagem do Repositório

> ℹ️ O repositório é **privado**. Você precisará de um Personal Access Token (PAT) do GitHub.
> 
> **Como gerar:** GitHub → Settings → Developer Settings → Personal Access Tokens → Fine-grained → `repo` scope

> **Alternativa:** Se o repositório for público, pode pular a etapa do token.

In [ ]:
from google.colab import userdata

# Opção A: Usar Colab Secrets (RECOMENDADO — mais seguro)
# Configure em: 🔑 ícone de chave na barra lateral → Add new secret
# Nome: GITHUB_TOKEN
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print('✅ Token carregado dos Colab Secrets')
except Exception:
    # Opção B: Inserir manualmente (menos seguro)
    GITHUB_TOKEN = input('Cole seu GitHub PAT aqui: ').strip()
    print('✅ Token inserido manualmente')

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/th3m1s-core/magistrado-themis-ai.git'
print('✅ URL do repositório configurada')

In [ ]:
import os

REPO_DIR = '/content/magistrado-themis-core'

if os.path.exists(REPO_DIR):
    print('📁 Repositório já existe. Fazendo pull...')
    !cd {REPO_DIR} && git pull
else:
    print('📥 Clonando repositório...')
    !git clone {REPO_URL} {REPO_DIR}

# Confirma estrutura
print('\n📂 Estrutura do projeto:')
!find {REPO_DIR} -not -path '*/.git/*' -maxdepth 3 | head -40
print('✅ Repositório pronto!')

In [ ]:
# Configura paths globais
import sys

REPO_DIR    = '/content/magistrado-themis-core'
SRC_DIR     = f'{REPO_DIR}/src'
DATASET_DIR = f'{REPO_DIR}/dataset'
MODELS_DIR  = f'{REPO_DIR}/models'
DATASET_PATH = f'{DATASET_DIR}/forensic_dataset.jsonl'
OUTPUT_DIR  = f'{MODELS_DIR}/magistrado-themis-qlora'

# Adiciona src ao path do Python
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

print(f'✅ REPO_DIR:     {REPO_DIR}')
print(f'✅ SRC_DIR:      {SRC_DIR}')
print(f'✅ DATASET_PATH: {DATASET_PATH}')
print(f'✅ OUTPUT_DIR:   {OUTPUT_DIR}')

---
## 🏭 Etapa 3 — GATE 1: Geração do Dataset Forense

Executa o **Active Learning Loop** para transformar eventos seed em pares `evento → laudo`.  
Se o dataset já existir no repositório (533KB, 125 entradas), esta etapa confirma e opcionalmente expande com augmentação.

In [ ]:
import json

# Inspeciona o dataset existente
if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    size_kb = os.path.getsize(DATASET_PATH) / 1024
    print(f'✅ Dataset existente: {len(lines)} entradas ({size_kb:.0f} KB)')

    # Mostra distribuição de labels
    labels = {}
    for line in lines:
        try:
            entry = json.loads(line)
            label = entry.get('_label', 'sem_label')
            labels[label] = labels.get(label, 0) + 1
        except:
            pass

    print('\n📊 Distribuição de labels:')
    for label, count in sorted(labels.items()):
        bar = '█' * count
        print(f'  {label:<40} {count:>3}x  {bar}')
else:
    print('⚠️ Dataset não encontrado. Executando GATE 1...')

In [ ]:
# Executa GATE 1 apenas se quiser expandir o dataset
# (o dataset já existente no repo é suficiente para treinar)

EXPAND_DATASET = False  # ← Mude para True se quiser gerar mais dados
AUGMENT_FACTOR = 3      # ← Variantes por evento seed (3 = ~54 entradas extras)

if EXPAND_DATASET:
    print('🔄 Expandindo dataset com Active Learning Loop...')
    os.chdir(REPO_DIR)
    !python src/magistrado/dataset_builder.py --batch --augment {AUGMENT_FACTOR} --output {DATASET_PATH}

    with open(DATASET_PATH, 'r') as f:
        total = len(f.readlines())
    print(f'✅ Dataset expandido: {total} entradas totais')
else:
    print('ℹ️  Usando dataset existente do repositório. EXPAND_DATASET=False')
    print(f'   Para expandir: defina EXPAND_DATASET = True')

---
## 🧠 Etapa 4 — GATE 2: Fine-Tuning QLoRA

Treina o **Qwen/Qwen2.5-3B-Instruct** com os adaptadores LoRA sobre o dataset forense.  
⏱️ **Tempo estimado:** ~45 min (T4, 3 épocas, 125 exemplos)

### Configurações QLoRA
| Parâmetro | Valor | Descrição |
|---|---|---|
| Quantização | 4-bit NF4 | Reduz VRAM para ~6GB |
| LoRA rank | 16 | Capacidade dos adaptadores |
| LoRA alpha | 32 | Escala de aprendizado |
| Target modules | q/k/v/o proj | Atenção completa |

In [ ]:
# 2A — Carregamento do Dataset
import json
from datasets import Dataset

def format_instruction(event: dict) -> str:
    """Formata evento como prompt de instruction-tuning para o Qwen."""
    metrics = event.get('metrics', {})
    return f"""Analise o seguinte evento forense do GuardTag e emita um laudo técnico-jurídico completo.

GTID: {event.get('gtid', 'N/A')}
Empresa: {event.get('empresa', 'N/A')}
Segmento: {event.get('segmento', 'N/A')}

Métricas de Autenticidade:
- Trust Score: {metrics.get('trust_score', 0):.1f}/100
- Similaridade Óptica (OFP): {metrics.get('optical_similarity', 0):.2f}
- Consistência RF/NFC: {metrics.get('rf_consistency', 0):.2f}
- Integridade do Selo (Tamper): {metrics.get('tamper_evidence', 0):.2f}

Timestamp: {event.get('timestamp', 'N/A')}

Emita um laudo forense completo com:
1. Conclusão objetiva (AUTÊNTICO / SUSPEITO / FRAUDADO)
2. Fundamentação técnica detalhada
3. Amparo legal (LGPD, CPC, Marco Civil, ICP-Brasil)
4. Recomendação de conduta operacional
5. Assinatura criptográfica (SHA-256)"""

def load_forensic_dataset(path: str) -> Dataset:
    examples = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                data = json.loads(line)
                inp = format_instruction(data['input'])
                out = data['output']
                examples.append({'input': inp, 'output': out})
            except Exception as e:
                print(f'  Linha ignorada: {e}')

    return Dataset.from_list(examples)

print('📂 Carregando dataset forense...')
dataset = load_forensic_dataset(DATASET_PATH)
print(f'✅ Dataset: {len(dataset)} exemplos carregados')
print(f'\n📄 Exemplo de input (primeiros 400 chars):')
print(dataset[0]['input'][:400])
print('\n📄 Output (primeiros 300 chars):')
print(dataset[0]['output'][:300])

In [ ]:
# 2B/2C/2D — Quantização 4-bit + Carregamento do Modelo + k-bit training
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LENGTH = 2048

print(f'📦 Carregando modelo: {MODEL_NAME}')
print('⏱️  Aguarde ~3-5 minutos para download...')

# 2B — Configuração BitsAndBytes 4-bit NF4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

# 2C — Carrega modelo base Qwen-2.5-3B
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

# 2D — Prepara para k-bit training
model = prepare_model_for_kbit_training(model)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'✅ Modelo carregado!')
print(f'   Parâmetros totais: {sum(p.numel() for p in model.parameters()):,}')
print(f'   VRAM usada: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# 2E — Injeção dos Adaptadores LoRA
from peft import LoraConfig, get_peft_model

print('🔧 Configurando adaptadores LoRA...')

lora_config = LoraConfig(
    r=16,                    # rank: equilíbrio capacidade/eficiência
    lora_alpha=32,           # alpha = 2x rank (prática padrão)
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

# 2E — Aplica LoRA no modelo
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'\n✅ LoRA configurado!')
print(f'   Parâmetros treináveis: {trainable:,} ({100*trainable/total:.2f}% do total)')

In [ ]:
# 3A/3B/3C — Formatação ChatML, Tokenização e Processamento em Batch

def tokenize_function(examples):
    """3A — Formata prompt ChatML e 3B — tokeniza o dataset."""
    texts = [
        f'<|im_start|>user\n{inp}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>'
        for inp, out in zip(examples['input'], examples['output'])
    ]

    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding='max_length',
        return_tensors=None,
    )

    # Labels = input_ids (causal LM)
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

print('🔤 Tokenizando dataset...')
# 3C — Processamento em batch
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
)

print(f'✅ Dataset tokenizado: {len(tokenized_dataset)} exemplos')
print(f'   Colunas: {tokenized_dataset.column_names}')
print(f'   Shape do primeiro exemplo: {len(tokenized_dataset[0]["input_ids"])} tokens')

In [ ]:
# 3D/3E — Configuração do Treinador e Execução do Fine-Tuning
from transformers import TrainingArguments, Trainer
import datetime

# ── Configurações de treinamento ──────────────────────────────────────────
# Ajuste conforme sua GPU:
#   T4  (16GB): batch_size=4, epochs=3  → ~45 min
#   A100(40GB): batch_size=8, epochs=5  → ~20 min
#   V100(16GB): batch_size=4, epochs=3  → ~50 min
EPOCHS     = 3      # ← Ajuste aqui
BATCH_SIZE = 4      # ← Reduza para 2 se houver OOM
LEARNING_RATE = 2e-4

print('⚙️  Configurações de treinamento:')
print(f'   Épocas:        {EPOCHS}')
print(f'   Batch size:    {BATCH_SIZE}')
print(f'   Learning rate: {LEARNING_RATE}')
print(f'   Output dir:    {OUTPUT_DIR}')

# 3D — TrainingArguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=LEARNING_RATE,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    optim='paged_adamw_32bit',
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    report_to='none',
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

print(f'\n🚀 Iniciando fine-tuning — {datetime.datetime.now().strftime("%H:%M:%S")}')
print('   (Acompanhe as métricas abaixo)')

# 3E — Executa treinamento
train_result = trainer.train()

print(f'\n✅ Fine-tuning concluído — {datetime.datetime.now().strftime("%H:%M:%S")}')
print(f'   Loss final:      {train_result.training_loss:.4f}')
print(f'   Passos totais:   {train_result.global_step}')
print(f'   Runtime:         {train_result.metrics.get("train_runtime", 0)/60:.1f} min')

In [ ]:
# 3F — Salvamento do Modelo Treinado
import json

print(f'💾 Salvando modelo em: {OUTPUT_DIR}')

# 3F — Salva model + tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Salva metadados do treino
meta = {
    'model_name': 'magistrado-themis-qlora-v1',
    'base_model': 'Qwen/Qwen2.5-3B-Instruct',
    'trained_at': datetime.datetime.now().isoformat(),
    'dataset_size': len(dataset),
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'lora_r': 16,
    'lora_alpha': 32,
    'final_loss': train_result.training_loss,
    'global_step': train_result.global_step,
    'owner': 'Symbeon Labs',
    'classification': 'CONFIDENCIAL - SEGREDO INDUSTRIAL',
}
with open(f'{OUTPUT_DIR}/training_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'\n📋 Arquivos salvos:')
!ls -lh {OUTPUT_DIR}
print('\n✅ Modelo salvo com sucesso!')

---
## 🧪 Etapa 5 — Testar o Modelo Treinado

Gera um laudo com o modelo fine-tunado e compara com o motor rule-based original.

In [ ]:
from transformers import pipeline
from peft import PeftModel

# Recarrega modelo treinado (LoRA merged)
print('🔄 Carregando modelo treinado para inferência...')

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
tuned_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
tuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)

print('✅ Modelo carregado para inferência!')

# Evento de teste (não visto no treino)
TEST_EVENT = {
    'gtid': 'GD-TEST-9999',
    'metrics': {
        'trust_score': 67.5,
        'optical_similarity': 0.74,
        'rf_consistency': 0.0,
        'tamper_evidence': 0.85
    },
    'empresa': 'Movida Locações de Veículos',
    'segmento': 'locadora',
    'timestamp': '2026-06-30T22:00:00Z'
}

prompt = format_instruction(TEST_EVENT)
full_prompt = f'<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n'

print('\n⚡ Gerando laudo com modelo fine-tunado...')
inputs = tuned_tokenizer(full_prompt, return_tensors='pt').to('cuda')
with torch.no_grad():
    outputs = tuned_model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tuned_tokenizer.eos_token_id,
    )

generated = tuned_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('\n' + '='*60)
print('  LAUDO GERADO PELO MODELO FINE-TUNADO')
print('='*60)
print(generated)

In [ ]:
# Avaliação qualitativa do laudo gerado
print('📊 Checklist de qualidade do laudo gerado:')

checks = {
    'Contém classificação (AUTÊNTICO/SUSPEITO/FRAUDADO)': any(x in generated for x in ['AUTÊNTICO', 'SUSPEITO', 'FRAUDADO']),
    'Contém referência legal (LGPD/CPC/Marco Civil)':     any(x in generated for x in ['LGPD', 'CPC', 'Marco Civil', 'ICP-Brasil']),
    'Contém Trust Score numérico':                        any(c.isdigit() for c in generated),
    'Contém similaridade óptica (OFP)':                   'OFP' in generated or 'óptica' in generated.lower(),
    'Contém assinatura criptográfica':                    'hash' in generated.lower() or '0x' in generated or 'sha' in generated.lower(),
    'Output > 500 chars (laudo completo)':                len(generated) > 500,
    'Linguagem técnico-jurídica formal':                  'relatoria' in generated.lower() or 'laudo' in generated.lower(),
}

passed = 0
for check, result in checks.items():
    icon = '✅' if result else '❌'
    print(f'  {icon} {check}')
    if result:
        passed += 1

score = passed / len(checks) * 100
print(f'\n🎯 Score de qualidade: {passed}/{len(checks)} ({score:.0f}%)')

if score >= 80:
    print('🏆 Excelente! Modelo pronto para GATE 3 (integração API).')
elif score >= 60:
    print('⚠️  Razoável. Considere mais épocas ou expandir dataset.')
else:
    print('❌ Abaixo do esperado. Execute mais épocas ou aumente dataset.')

---
## 💾 Etapa 6 — Salvar e Baixar o Modelo

Três opções: **download direto**, **Google Drive** ou **push para HuggingFace Hub**.

In [ ]:
# Opção A: Salvar no Google Drive (RECOMENDADO)
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_OUTPUT = '/content/drive/MyDrive/magistrado-themis-models/qlora-v1'
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

    print(f'📁 Copiando modelo para Google Drive...')
    !cp -r {OUTPUT_DIR}/* {DRIVE_OUTPUT}/

    print(f'✅ Modelo salvo em: {DRIVE_OUTPUT}')
    !ls -lh {DRIVE_OUTPUT}

In [ ]:
# Opção B: Download direto como ZIP
DOWNLOAD_ZIP = False  # ← Mude para True se quiser baixar agora

if DOWNLOAD_ZIP:
    from google.colab import files
    import zipfile

    ZIP_PATH = '/content/magistrado-themis-qlora-v1.zip'
    print(f'📦 Comprimindo modelo ({OUTPUT_DIR})...')

    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files_list in os.walk(OUTPUT_DIR):
            for file in files_list:
                fp = os.path.join(root, file)
                zf.write(fp, os.path.relpath(fp, OUTPUT_DIR))

    size_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f'✅ ZIP criado: {size_mb:.0f} MB')
    print(f'📥 Iniciando download...')
    files.download(ZIP_PATH)

In [ ]:
# Opção C: Push para HuggingFace Hub (privado)
# Requer HF_TOKEN nos Colab Secrets
PUSH_TO_HF = False  # ← Mude para True se quiser hospedar no HF Hub

if PUSH_TO_HF:
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except:
        HF_TOKEN = input('Cole seu HuggingFace Token: ').strip()

    HF_REPO_ID = 'th3m1s-core/magistrado-themis-qlora-v1'

    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)

    # Cria repositório privado
    api.create_repo(repo_id=HF_REPO_ID, private=True, exist_ok=True)

    # Push model
    tuned_model.push_to_hub(HF_REPO_ID, token=HF_TOKEN, private=True)
    tuned_tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN, private=True)

    print(f'✅ Modelo publicado (privado): https://huggingface.co/{HF_REPO_ID}')

---
## ✅ Pipeline Concluído!

| Etapa | Status |
|---|---|
| GATE 1 — Dataset forense | ✅ |
| GATE 2 — Fine-tuning QLoRA | ✅ |
| Modelo testado | ✅ |
| Modelo salvo | ✅ |

### Próximos Passos (GATE 3)

1. **Carregar o modelo no servidor FastAPI** (`guarddrive-intelligence-hub/backend`)
2. **Ativar endpoint** `POST /api/magistrado/laudo` com o modelo fine-tunado
3. **Testes jurídicos** com RS Advogados
4. **Ajustar linguagem** conforme feedback jurídico
5. **GATE 4** — Depósito de patente INPI

---
*Magistrado Themis | Symbeon Labs*  
*CONFIDENCIAL — SEGREDO INDUSTRIAL*